# TASK 2 · Customer Segmentation Analysis

**Objective:** Segment customers into distinct groups based on purchasing behaviour using **RFM-style features and K-Means clustering**.

**Tech Stack:** Python, pandas, scikit-learn, matplotlib, seaborn, Jupyter Notebook


## 1. Load and Inspect the Dataset

The uploaded CSV contains **500 rows and 7 columns**.

Detected fields:
- Customer ID: `not detected`
- Date: `not detected`
- Monetary field: `Yearly Amount Spent`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("/mnt/data/data.csv")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)


### Data cleaning

Exact duplicate rows are removed. Numeric fields are converted where appropriate, and date fields are parsed with invalid values converted to missing. Missing behavioural values are handled during feature engineering.


In [ ]:
clean = df.copy()
clean.columns = [c.strip() for c in clean.columns]
clean = clean.drop_duplicates().copy()

customer_col = None
date_col = None
amount_col = 'Yearly Amount Spent'
qty_col = None
price_col = None

if date_col:
    clean[date_col] = pd.to_datetime(clean[date_col], errors="coerce")

for col in [amount_col, qty_col, price_col]:
    if col and col in clean:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

if amount_col:
    clean["_amount"] = clean[amount_col]
elif qty_col and price_col:
    clean["_amount"] = clean[qty_col] * clean[price_col]
else:
    nums = clean.select_dtypes(include="number").columns.tolist()
    if not nums:
        raise ValueError("No numeric field is available for monetary analysis.")
    clean["_amount"] = clean[nums[0]]

if customer_col:
    clean["_customer"] = clean[customer_col].astype(str)
    print("Using the detected customer identifier.")
else:
    clean["_customer"] = np.arange(len(clean)).astype(str)
    print("WARNING: No customer ID detected. Rows are treated as customer proxies.")


## 2. Customer-Level Behavioural Features

The preferred framework is **RFM**:
- **Recency:** days since the customer's latest purchase.
- **Frequency:** number of transactions.
- **Monetary:** total observed customer spend.

We also calculate **Average Purchase Value (APV)**. If no usable transaction date exists, Recency is unavailable and the notebook uses Frequency, Monetary and APV instead.


In [ ]:
if date_col:
    reference_date = clean[date_col].max()
    customer_df = clean.groupby("_customer").agg(
        Frequency=("_customer", "size"),
        Monetary=("_amount", "sum"),
        Average_Purchase_Value=("_amount", "mean"),
        Last_Purchase_Date=(date_col, "max")
    ).reset_index()
    customer_df["Recency"] = (reference_date - customer_df["Last_Purchase_Date"]).dt.days
else:
    customer_df = clean.groupby("_customer").agg(
        Frequency=("_customer", "size"),
        Monetary=("_amount", "sum"),
        Average_Purchase_Value=("_amount", "mean")
    ).reset_index()
    customer_df["Recency"] = np.nan

for col in ["Frequency","Monetary","Average_Purchase_Value","Recency"]:
    if customer_df[col].notna().any():
        customer_df[col] = pd.to_numeric(customer_df[col], errors="coerce")
        customer_df[col] = customer_df[col].fillna(customer_df[col].median())

display(customer_df.head())


## 3. Descriptive Statistics

Customer Lifetime Value (CLV) is represented here by **observed cumulative monetary value**. This is a CLV proxy rather than a predictive lifetime-value model.


In [ ]:
stats = customer_df[["Average_Purchase_Value","Frequency","Monetary"]].agg(
    ["mean","median","std"]
).T
stats["mode"] = [
    customer_df[c].mode().iloc[0] if not customer_df[c].mode().empty else np.nan
    for c in ["Average_Purchase_Value","Frequency","Monetary"]
]
stats = stats[["mean","median","mode","std"]]
stats.columns = ["Mean","Median","Mode","Standard Deviation"]
display(stats)


## 4. Feature Selection and Standardisation

The clustering model uses 2–3 key behavioural features. RFM is preferred when a date exists; otherwise, the best available behavioural proxies are selected.

Because K-Means uses distance calculations, **StandardScaler** is applied before clustering so one feature does not dominate simply because of its scale.


In [ ]:
if customer_df["Recency"].notna().any():
    features = ["Recency","Frequency","Monetary"]
else:
    features = ["Frequency","Monetary","Average_Purchase_Value"]

X = customer_df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Clustering features:", features)
display(X.describe().T)


## 5. Elbow Method

The Elbow Method evaluates K-Means inertia for several values of K. Silhouette score is also calculated as a secondary diagnostic, and the best silhouette score is used to select a reproducible K.


In [ ]:
max_k = min(10, max(2, len(X_scaled)-1))
ks = list(range(2, max_k+1))
inertias, silhouettes = [], []

for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(9,5))
plt.plot(ks, inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.xticks(ks)
plt.grid(alpha=.25)
plt.show()

score_df = pd.DataFrame({"K":ks, "Silhouette Score":silhouettes})
display(score_df)
optimal_k = int(score_df.loc[score_df["Silhouette Score"].idxmax(), "K"])
print("Selected K:", optimal_k)


## 6. K-Means Clustering

The selected model assigns each customer to a behavioural segment.


In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customer_df["Cluster"] = kmeans.fit_predict(X_scaled)
display(customer_df.head())


## 7. Cluster Visualisation

Two different feature combinations are plotted to make the segments easier to interpret from multiple behavioural perspectives.


In [ ]:
x1, y1 = ("Recency","Monetary") if "Recency" in features else (features[0],features[1])
plt.figure(figsize=(9,6))
sns.scatterplot(data=customer_df, x=x1, y=y1, hue="Cluster", palette="tab10", s=70)
plt.title(f"Clusters: {x1} vs {y1}")
plt.show()


In [ ]:
x2, y2 = ("Frequency","Average_Purchase_Value")
plt.figure(figsize=(9,6))
sns.scatterplot(data=customer_df, x=x2, y=y2, hue="Cluster", palette="tab10", s=70)
plt.title(f"Clusters: {x2} vs {y2}")
plt.show()


### Observation

The first chart emphasises customer value and activity timing, while the second separates customers by purchase frequency and transaction size. Consistent separation across both views gives greater confidence that the clusters represent useful behavioural differences.


## 8. Cluster Profiling

Mean feature values and segment sizes are calculated for every cluster. The resulting profiles can be translated into customer types for marketing.


In [ ]:
profile_cols = ["Recency","Frequency","Average_Purchase_Value","Monetary"]
profile = customer_df.groupby("Cluster")[profile_cols].mean().round(2)
profile["Customer_Count"] = customer_df["Cluster"].value_counts().sort_index()
profile["Share_%"] = (profile["Customer_Count"] / len(customer_df) * 100).round(2)
display(profile)


In [ ]:
overall = customer_df[profile_cols].mean()
relative = profile[profile_cols].div(overall).round(2)
display(relative)


### Customer type interpretation

- **High-Value Loyal:** high frequency and high monetary contribution.
- **Active / Promising:** relatively recent activity with room to increase value.
- **Low-Engagement:** low frequency and low monetary contribution.
- **At-Risk / Lapsed:** relatively high Recency (longer time since purchase).

These labels are business interpretations; the numerical profile should remain the primary evidence.


In [ ]:
freq_med = customer_df["Frequency"].median()
mon_med = customer_df["Monetary"].median()
rec_med = customer_df["Recency"].median()

def segment_label(row):
    if row["Frequency"] >= freq_med and row["Monetary"] >= mon_med:
        return "High-Value Loyal"
    if row["Recency"] <= rec_med and row["Monetary"] >= mon_med:
        return "Active / Promising"
    if row["Recency"] > rec_med and row["Monetary"] < mon_med:
        return "At-Risk / Lapsed"
    return "Low-Engagement"

profile["Customer_Type"] = profile.apply(segment_label, axis=1)
display(profile)


## 9. Customers per Cluster

Segment size matters for campaign planning. A small but high-value segment may deserve more retention investment than a much larger low-value group.


In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=customer_df, x="Cluster", hue="Cluster", palette="tab10", legend=False)
plt.title("Number of Customers per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.show()


## 10. Insights and Marketing Actions

| Segment | Recommended marketing action |
|---|---|
| **High-Value Loyal** | VIP benefits, early access, premium bundles and loyalty rewards; minimise unnecessary blanket discounts. |
| **Active / Promising** | Cross-sell complementary products, personalised recommendations and loyalty-program enrolment. |
| **Low-Engagement** | Product discovery campaigns, personalised low-cost incentives and educational/onboarding content. |
| **At-Risk / Lapsed** | Win-back emails, reminders, time-limited incentives and reactivation journeys. |

### Actionable recommendations
1. **Protect high-value customers** with retention-first campaigns rather than broad discounts.
2. **Convert promising customers** through cross-selling, bundles and loyalty rewards.
3. **Reactivate lapsed customers** using personalised win-back campaigns based on their historical value.
4. **Allocate campaign budgets by segment size and value**, not simply by number of customers.
5. **Re-run segmentation periodically** because customer behaviour changes over time.


## 11. Conclusion

K-Means provides a practical way to turn transaction behaviour into actionable customer segments. The most useful output is not simply the cluster number, but the **behavioural profile behind each cluster** and the marketing action attached to it.

**Important limitation:** if the source data lacks a true Customer ID or transaction date, genuine longitudinal RFM analysis is not possible. In that situation this notebook uses transparent proxies and should be upgraded with customer-level historical transaction data for production use.
